# E1 Walkthrough: Universe, Returns and the Hygiene Ledger

Mechanistic explanation of the E1 data layer, reproduced by hand so
every downstream sprint starts from numbers I have verified. Every
number printed below is recomputed here with plain arithmetic, not
read from an output file, except where the section says it reads the
artifact to reconcile to it.

Research questions. Academic: what is a return, and which definition
(simple, log, excess) is correct for which operation. Practitioner:
can I trust the prices, the universe and the risk-free rate underneath
every number this book will ever show. Research: is a free,
survivorship-affected universe good enough for factor-model research,
and how large is the bias in basis points per year.

Intuition. Log returns add through time and simple returns add across
a portfolio; mixing them is a bug that survives for years because it
is small on any one day. Daily market returns have kurtosis far above
3, so a Gaussian risk number is a floor. A Sharpe ratio of 1 measured
over three years carries a standard error near 0.7, so an eyeballed
Sharpe is a hypothesis. A universe built from today's members is a
universe of winners.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
prices = pd.read_parquet(ROOT / "data/raw/prices.parquet")
returns_art = pd.read_parquet(ROOT / "data/processed/returns.parquet")
members = pd.read_parquet(ROOT / "data/processed/universe_membership.parquet")
ff = pd.read_parquet(ROOT / "data/raw/factors_ff.parquet")
events = pd.read_parquet(ROOT / "data/processed/events.parquet")
results = json.loads((ROOT / "sprints/E1/RESULTS.json").read_text())

NAMES = ["AAPL", "XOM", "JPM"]
DATES = [pd.Timestamp(d) for d in ["2026-06-15", "2026-06-16", "2026-06-17"]]
print("artifacts loaded")

## 1. Returns by hand

For AAPL, XOM and JPM on three consecutive business days, compute
simple, log and excess returns from adjusted close and the daily RF
with plain arithmetic. Each symbol is labeled INPUT (file, column) or
OUTPUT (file, column).

In [ ]:
rows = []
for name in NAMES:
    adj = prices.xs(name, level="ticker")["adj_close"]  # INPUT: prices.parquet, adj_close
    for d in DATES:
        prev = adj.loc[d - pd.Timedelta(days=7): d - pd.Timedelta(days=1)].iloc[-1]
        p_t = adj.loc[d]
        rf_t = ff.loc[d, "rf"]  # INPUT: factors_ff.parquet, rf
        r = p_t / prev - 1.0
        g = np.log(p_t / prev)
        excess = r - rf_t
        stored = returns_art.loc[(d, name)]  # OUTPUT: returns.parquet, r g excess
        rows.append((name, d, r, stored["r"], g, stored["g"], excess, stored["excess"]))

table = pd.DataFrame(
    rows,
    columns=["ticker", "date", "r_hand", "r_stored", "g_hand", "g_stored",
             "excess_hand", "excess_stored"],
)
for col in ["r", "g", "excess"]:
    diff = (table[f"{col}_hand"] - table[f"{col}_stored"]).abs().max()
    print(f"max |{col}_hand - {col}_stored| = {diff:.3e}")
    assert diff < 1e-10
print(table.round(8).to_string(index=False))

## 2. Aggregation

Log returns add over time and simple returns add across an
equal-weight portfolio. Show both with numbers, then show where each
approximation breaks.

In [ ]:
for name in NAMES:
    adj = prices.xs(name, level="ticker")["adj_close"]
    p0 = adj.loc[DATES[0] - pd.Timedelta(days=7): DATES[0] - pd.Timedelta(days=1)].iloc[-1]
    pT = adj.loc[DATES[-1]]
    g = returns_art.xs(name, level="ticker")["g"].loc[DATES]
    exact = np.log(pT / p0)
    print(name, f"sum of log returns = {g.sum():.10f}, ln(P_T/P_0) = {exact:.10f}, diff = {abs(g.sum() - exact):.2e}")

print()
r = returns_art["r"].unstack("ticker")[NAMES].loc[DATES]
ew = r.mean(axis=1)
print("equal-weight portfolio: simple returns add across assets")
print(ew.round(8).to_string())

print()
print("where each approximation breaks:")
big = pd.Series([0.20, -0.15, 0.30])  # synthetic big daily moves
print("log vs simple for big moves: r - g = r^2/2 + ...")
print(pd.DataFrame({"r": big, "g": np.log1p(big), "r_minus_g": big - np.log1p(big)}).round(6).to_string(index=False))
print("simple returns across assets hold exactly: sum of weights times r equals the portfolio return")
print("log returns do NOT add across assets: log(sum_i w_i (1+r_i)) is not sum_i w_i g_i")
w = np.array([0.5, 0.3, 0.2])
port_log = np.log((w * (1 + big)).sum())
sum_wg = (w * np.log1p(big)).sum()
print(f"portfolio log return = {port_log:.6f}, weighted sum of log returns = {sum_wg:.6f}, gap = {port_log - sum_wg:.6f}")

## 3. Stylized facts with numbers

Kurtosis of daily returns and autocorrelation of r at lag 1 and of
r squared at lags 1, 5, 21, for the three names and the equal-weight
universe. Recomputed by hand and reconciled to the library.

In [ ]:
from efb import returns as retmod

def acf_hand(x, lag):
    v = x.dropna().to_numpy()
    a = v[:-lag]
    b = v[lag:]
    num = ((a - a.mean()) * (b - b.mean())).sum()
    den = np.sqrt(((a - a.mean()) ** 2).sum() * ((b - b.mean()) ** 2).sum())
    return num / den

def kurt_hand(x):
    v = x.dropna().to_numpy()
    n = len(v)
    m2 = ((v - v.mean()) ** 2).mean()
    m4 = ((v - v.mean()) ** 4).mean()
    excess = (n - 1) / ((n - 2) * (n - 3)) * ((n + 1) * m4 / m2**2 - 3 * (n - 1))
    return excess + 3.0

ew = retmod.equal_weight_universe_return(returns_art, members).dropna()
for label, series in [(n, returns_art.xs(n, level="ticker")["r"]) for n in NAMES] + [("ew", ew)]:
    hand = {
        "kurtosis": kurt_hand(series),
        "acf_r_lag1": acf_hand(series, 1),
        "acf_r2_lag1": acf_hand(series**2, 1),
        "acf_r2_lag5": acf_hand(series**2, 5),
        "acf_r2_lag21": acf_hand(series**2, 21),
    }
    lib = retmod.stylized_facts(series)
    diffs = {k: abs(hand[k] - lib[k]) for k in hand}
    print(label, {k: round(v, 4) for k, v in hand.items()})
    assert max(diffs.values()) < 1e-8, diffs


## 4. Sharpe with standard error

Compute the Sharpe ratio, the i.i.d. standard error and the Lo (2002)
standard error for the FF market factor step by step, and explain why
the two standard errors differ and which way.

In [ ]:
from efb import perf

x = ff["mkt_rf"].dropna()
T = len(x)
sr = x.mean() / x.std(ddof=1)
se_iid = np.sqrt((1 + sr**2 / 2) / T)
print(f"T = {T}, SR daily = {sr:.6f}, SR annualized = {sr * np.sqrt(252):.4f}")
print(f"SE iid daily = {se_iid:.6f}, annualized = {se_iid * np.sqrt(252):.4f}")

q = 5
def ww(values, q):
    z = values - values.mean()
    var = (z * z).sum() / len(z)
    out, terms = 0.0, []
    for k in range(1, q + 1):
        wk = 1 - k / (q + 1)
        rho = (z[:-k] * z[k:]).sum() / len(z) / var
        terms.append((k, wk, rho))
        out += wk * rho
    return out, terms

sum_rho, terms_rho = ww(x.to_numpy(), q)
sum_phi, terms_phi = ww((x**2).to_numpy(), q)
print("autocorrelations of x:")
for k, wk, rho in terms_rho:
    print(f"  lag {k}: weight {wk:.3f}, rho_k = {rho:+.4f}")
print("autocorrelations of x squared:")
for k, wk, phi in terms_phi:
    print(f"  lag {k}: weight {wk:.3f}, phi_k = {phi:+.4f}")
A = 1 + 2 * sum_rho
B = 1 + 2 * sum_phi
se_lo = np.sqrt((A + (sr**2 / 2) * B) / T)
print(f"A = {A:.4f}, B = {B:.4f}, SR^2/2 = {sr**2/2:.6f}")
print(f"SE Lo daily = {se_lo:.6f}, annualized = {se_lo * np.sqrt(252):.4f}")
print(f"ratio Lo / iid = {se_lo / se_iid:.4f}")
assert np.isclose(se_lo, perf.sharpe_se_lo2002(x, q=5))
assert np.isclose(se_iid, perf.sharpe_se_iid(x))
print("Why: the mean term A uses the autocorrelation of returns, which is")
print("negative at lag 1 here, shrinking the mean's variance; the volatility")
print("term B uses the autocorrelation of squared returns, which is strongly")
print("positive (volatility clustering), but it enters weighted by SR^2/2,")
print("which is tiny for SR around 0.048. Net effect: the Lo standard error")
print("is 8 percent smaller than the iid one for this series. For a series")
print("with positive lag-1 autocorrelation the correction runs the other way.")

## 5. Survivorship

The F1.5 number, and the naive buy-all-members backtest versus the FF
market return, plotted, with the annualized gap in basis points.

In [ ]:
import plotly.graph_objects as go

f15 = results["criteria"]["F1.5"]["stored_numbers"]
print(f"fraction of deleted members with recoverable history = {f15['fraction_recovered']:.4f}")

current = sorted(set(members.columns[members.iloc[-1]]))
wide = returns_art["r"].unstack("ticker")
naive = wide[[c for c in current if c in wide.columns]].mean(axis=1).dropna()
pit = retmod.equal_weight_universe_return(returns_art, members).dropna()
mkt = ff["mkt_rf"] + ff["rf"]
common = pd.concat([naive.rename("naive"), pit.rename("pit"), mkt.rename("mkt")], axis=1, join="inner").dropna()

def annualized_bp(a, b):
    return (a.mean() - b.mean()) * 252 * 1e4

print(f"naive minus pit      = {annualized_bp(common['naive'], common['pit']):.1f} bp per year")
print(f"naive minus market   = {annualized_bp(common['naive'], common['mkt']):.1f} bp per year")
print(f"pit minus market     = {annualized_bp(common['pit'], common['mkt']):.1f} bp per year")

wealth = (1 + common).cumprod()
fig = go.Figure()
for col in ["naive", "pit", "mkt"]:
    fig.add_scatter(x=wealth.index, y=wealth[col], name=col, mode="lines")
fig.update_layout(title="Cumulative wealth: buy-all-current-members vs point-in-time vs FF market",
                  yaxis_type="log", height=420)
fig.show(renderer="notebook")

## 6. One section per F1.x criterion

Threshold, stored number, verdict, and what a failure would have meant.

In [ ]:
meaning = {
    "F1.1": "A source below 95 percent would mean the data layer rests on an unverified source; the design must adapt around it.",
    "F1.2": "Interior NaNs would mean coverage holes that later regressions would silently inherit; they must be documented or the data is not usable as-is.",
    "F1.3": "A correlation below 0.95 would mean a date-alignment or adjustment bug: the data layer would be wrong before any model exists.",
    "F1.4": "A gap above 1 bp would mean adjusted close and the dividend series disagree: a missed split or dividend would corrupt every return.",
    "F1.5": "A fraction below 70 percent forces the ledger to record the survivorship bias; a bias above 200 bp per year means long-only backtests carry a mandatory caveat and long/short constructions are preferred.",
}
for fid in ["F1.1", "F1.2", "F1.3", "F1.4", "F1.5"]:
    c = results["criteria"][fid]
    print(f"{fid}: threshold {c['threshold']} | stored {c.get('stored_number', c.get('stored_numbers'))} | verdict {c['verdict']}")
    print(f"  what a failure would have meant: {meaning[fid]}")

## 7. Dashboard D0: panel to parquet column map

Each panel of D0 Data Health reads exactly these files and columns; the
dashboard never recomputes a number.

In [ ]:
mapping = pd.DataFrame(
    [
        ("Coverage heatmap (ticker x month)", "data/raw/prices.parquet", "adj_close"),
        ("Missing tickers (last 5 business days)", "data/processed/returns.parquet", "r"),
        ("Stale counts", "data/processed/returns.parquet", "stale"),
        ("Universe size over time", "data/processed/universe_membership.parquet", "boolean columns, row sums"),
        ("Additions and deletions", "data/processed/events.parquet", "event_type in (added, removed)"),
        ("Corporate-action and outlier log", "data/processed/events.parquet", "event_type, detail"),
        ("Hygiene Ledger rendered in-app", "docs/hygiene_ledger.md", "markdown"),
        ("Data version in the sidebar", "data/VERSION.json", "artifacts.<name>.sha256"),
    ],
    columns=["panel", "file", "column"],
)
print(mapping.to_string(index=False))

## 8. Credit port note

What survives unchanged when this data layer is ported to credit, and
what gets re-specified.

Survives unchanged. The excess-return definition (total return minus a
risk-free leg), the Sharpe ratio with its i.i.d. and Lo (2002)
standard errors, the annualization convention, max drawdown, hit rate
and slugging, the hygiene ledger structure (missing-data policy, stale
price detection, outlier flags with a threshold, point-in-time flags
per field), and the dashboard's rule that tabs read artifacts and
never recompute.

Re-specified. The return definition becomes spread or
excess-over-duration-matched-Treasury return instead of price return;
the universe membership source becomes index constituent files instead
of the Wikipedia changes table; the risk-free leg is a matched
Treasury rather than the daily T-bill rate; adjusted close and its
corporate-action audit are replaced by bond price provenance (clean
price plus accrued interest, per index convention); and the
survivorship measurement re-runs against the credit index's own
historical membership files.